# Step 2: Data Cleaning

Goal: Clean the data and prepare it for enrichment.

Rules:
- Keep all respondents (don't drop rows)
- Fix data types
- Split multi-select fields
- Keep missing values as null

**Before starting:** Create the `my_work` folder if it doesn't exist:
```bash
mkdir -p my_work
```

In [ ]:
import pandas as pd
import json
import os

In [ ]:
# Create my_work folder if it doesn't exist
os.makedirs("../my_work", exist_ok=True)

In [ ]:
df = pd.read_csv("../data/developer_ai_learning_raw.csv")

In [ ]:
# Check for duplicates
df.duplicated().sum()

In [ ]:
# Remove duplicates if any
df = df.drop_duplicates()

In [ ]:
# Convert YearsCode to nullable integer
df['YearsCode'] = df['YearsCode'].astype('Int64')

In [ ]:
# Function to split multi-select
def split_multiselect(value):
    if pd.isna(value):
        return None
    return value.split(";")

In [ ]:
# Split LearnCode
df['LearnCode'] = df['LearnCode'].apply(split_multiselect)

In [ ]:
# Split AILearnHow
df['AILearnHow'] = df['AILearnHow'].apply(split_multiselect)

In [ ]:
# Verify
df.shape

In [ ]:
df['YearsCode'].dtype

In [ ]:
df['LearnCode'].head()

In [ ]:
# Save to JSONL
with open("../my_work/cleaned_data.jsonl", "w") as f:
    for _, row in df.iterrows():
        record = row.to_dict()
        
        # Convert Int64 to int or None
        if pd.notna(record['YearsCode']):
            record['YearsCode'] = int(record['YearsCode'])
        else:
            record['YearsCode'] = None
        
        # Convert NaN to None
        for key, value in record.items():
            if not isinstance(value, list) and pd.isna(value):
                record[key] = None
        
        f.write(json.dumps(record) + "\n")

## Next Step

Run validation:
```bash
cd Day4_Project
python3 validation/check_step1_cleaning.py my_work/cleaned_data.jsonl
```